# TRM Sudoku-Extreme 复现分析

只展示本机真实产物。论文 87.4% 为参考，不是本次已取得成绩。源码版本 c01103738605ba39d1430519b1ee0c62f4c707f8；训练 1K 原题、每题 1000 个增强，验证与官方测试隔离。最终完整测试必须 n=422786 且 complete=true。

In [ ]:
from pathlib import Path
import json
import numpy as np
ROOT = Path.cwd()
if ROOT.name == 'trm': ROOT = ROOT.parent
if not (ROOT/'runs').exists(): ROOT = Path('/workspace/project')
RUN = ROOT/'runs/trm/sudoku_mlp_seed0'
manifest = json.loads((ROOT/'data/manifest.json').read_text())
print({k: manifest[k] for k in ['data_sha','seed','n_train','n_val','n_test','num_aug']})
print(json.loads((RUN/'status.json').read_text()) if (RUN/'status.json').exists() else '尚未开始正式训练')

In [ ]:
rows = [json.loads(x) for x in (RUN/'train.jsonl').read_text().splitlines()] if (RUN/'train.jsonl').exists() else []
print(rows[-1] if rows else '暂无训练日志')
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    for ax,key in zip(axes,['train_cell_accuracy','train_exact_accuracy']):
        ax.plot([r['step'] for r in rows],[r[key] for r in rows]); ax.set(xlabel='Optimizer update',ylabel=key)
    plt.show()
except ImportError:
    print('可选绘图依赖 matplotlib 未安装；数值日志不受影响。')

In [ ]:
validations = []
for p in sorted(RUN.glob('val_step_*/metrics.json')):
    m = json.loads(p.read_text()); validations.append((p.parent.name,m['exact_accuracy'],m['blank_cell_accuracy']))
print('Validation:',validations)
p = RUN/'official_test_final/metrics.json'
if p.exists():
    m = json.loads(p.read_text())
    print(m)
    if m['complete'] and m['n']==422786:
        print('Full official test complete. Exact accuracy:',format(m['exact_accuracy'],'.2%'))
    else:
        print('部分测试，不构成完整复现。')
else:
    print('尚无完整官方测试结果。')

In [ ]:
import numpy as np
p = RUN/'official_test_final/predictions.npy'
if p.exists():
    pred = np.load(p,mmap_mode='r')
    progress = json.loads((p.parent/'progress.json').read_text())['processed']
    q = np.load(ROOT/'data/q_test.npy',mmap_mode='r')
    s = np.load(ROOT/'data/s_test.npy',mmap_mode='r')
    for i in range(min(progress,3)):
        print('Question',q[i].reshape(9,9),'Prediction',pred[i].reshape(9,9),'Truth',s[i].reshape(9,9),sep='\n')

## 复现边界

- 官方构建脚本没有固定 NumPy seed，也未给出论文的 1K 原题索引。本项目固定 seed=0 并保存索引，属于同协议重新训练，不能宣称逐题等同作者划分。
- 有效 batch=768，梯度累积只用于适配 16GB。随机抽样与 BF16 累加次序可能导致数值差异。
- 官方 README 为 50K epochs，论文正文为 60K；本次执行 README 配方。实际 10 个 5000-epoch 窗口共 65100 次更新。
- 周期评估使用独立训练源验证集，官方测试只在最终 checkpoint 完成后运行，不用于挑选 checkpoint。
- 所有预测为 16 轮原始 argmax，无数独求解器、clue 覆盖或答案回填。
- trm_ode/ 暂无实现。